# 01 - Exploratory Data Analysis

Two data sources:
1. **Booking.com 515K Hotel Reviews** (Kaggle) - labeled training data (see `data/raw/README.md` for the download and the label rule).
2. **`hotel_reviews.csv`** - 658 real unlabeled reviews of two NYC hotels; our stand-in for production traffic.

Run this after `make data` (preprocessing) if you want the processed splits too.

## Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn

# Every path below is repo-root relative; allow running from notebooks/ too.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sn.set_theme(style="whitegrid")
pd.options.display.max_colwidth = 120

## 1. Raw Booking.com data

In [ ]:
RAW = "data/raw/Hotel_Reviews.csv"   # from Kaggle - see data/raw/README.md
df = pd.read_csv(RAW)
print(df.shape)
df[["Hotel_Name", "Reviewer_Score", "Positive_Review", "Negative_Review"]].head(3)

In [ ]:
# How unambiguous is the two-field schema?
# The raw fields carry leading/trailing whitespace, so strip before comparing -
# exactly what reviewnlp.data.preprocess does. Skip the strip and every row
# looks "mixed", which tells the opposite of the true story.
from reviewnlp.data.preprocess import NO_NEGATIVE, NO_POSITIVE

pos = df["Positive_Review"].astype(str).str.strip()
neg = df["Negative_Review"].astype(str).str.strip()

pos_only = ((pos != NO_POSITIVE) & (neg == NO_NEGATIVE)).sum()
neg_only = ((pos == NO_POSITIVE) & (neg != NO_NEGATIVE)).sum()
mixed    = ((pos != NO_POSITIVE) & (neg != NO_NEGATIVE)).sum()
empty    = ((pos == NO_POSITIVE) & (neg == NO_NEGATIVE)).sum()

print(f"positive-only={pos_only:,}  negative-only={neg_only:,}  "
      f"mixed={mixed:,}  empty={empty:,}  ->  usable={pos_only + neg_only:,}")

pd.DataFrame({"rows": [pos_only, neg_only, mixed, empty]},
             index=["positive-only", "negative-only", "mixed (dropped)", "empty"]).plot(
    kind="barh", figsize=(7, 3), legend=False, title="Booking schema: label sources")
plt.tight_layout(); plt.show()

In [ ]:
# Reviewer score distribution (we do NOT use it for labels - shown for context)
df["Reviewer_Score"].plot(kind="hist", bins=30, figsize=(7, 3), title="Reviewer_Score distribution")
plt.tight_layout(); plt.show()

## 2. Processed splits (after `make data`)

In [ ]:
if all(os.path.exists(f"data/processed/{s}.parquet") for s in ("train", "dev", "test")):
    for s in ("train", "dev", "test"):
        d = pd.read_parquet(f"data/processed/{s}.parquet")
        print(f"{s:>5}: {len(d):>7,} rows | pos={int((d['label']=='positive').sum()):>7,} "
              f"neg={int((d['label']=='negative').sum()):>7,} | avg chars={d['text'].str.len().mean():.0f}")
else:
    print("processed splits not found - run `make data` first")

In [ ]:
# Review length profile - informs max_length / max_tokens choices
if os.path.exists("data/processed/train.parquet"):
    d = pd.read_parquet("data/processed/train.parquet")
    lens = d["text"].str.len()
    words = d["text"].str.split().str.len()
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    lens.clip(upper=1500).plot(kind="hist", bins=50, ax=axes[0], title="characters (clipped at 1500)")
    words.clip(upper=300).plot(kind="hist", bins=50, ax=axes[1], title="words (clipped at 300)")
    plt.tight_layout(); plt.show()
    print(f"p50/p95/p99 words: {words.quantile([.5,.95,.99]).round(0).to_dict()}")

## 3. The unlabeled production file (hotel_reviews.csv)

In [ ]:
own = pd.read_csv("data/raw/hotel_reviews.csv")
print(own.shape, "| hotels:", own["Hotel"].unique().tolist())
own["Review"].str.len().plot(kind="hist", bins=40, figsize=(7, 3),
                             title="hotel_reviews.csv: review length (chars)")
plt.tight_layout(); plt.show()
own["Hotel"].value_counts().plot(kind="barh", figsize=(5, 2), title="reviews per hotel")
plt.tight_layout(); plt.show()

## Takeaways
- The two-field schema yields hundreds of thousands of *unambiguous* labels - no score threshold needed.
- Reviews are short (most < 300 words) so `max_length=256` (encoders) and `max_tokens=220` (BiLSTM) are safe.
- The unlabeled file is small but realistic; it feeds the LLM annotation demo and the API examples.